In [1]:
import gwsus as sus
import gwsus.utils as utils
import gwsus.materials as materials
import gwsus.shapes3d as shapes
from gwsus import info
from gwsus import anime

<IPython.core.display.Javascript object>

In [2]:
import sympy as sp
import mpmath as mpm # for multiple precision numerical mathematics

In [3]:
def is_symmetric(A):
    return all(
        A[i, j] == A[j, i]
        for i in range(A.rows)
        for j in range(i)
    )


def find_zero_rows(A):
    """Return the indices of exactly zero rows in matrix A."""
    return [
        i for i in range(A.rows)
        if all(A[i, j] == 0 for j in range(A.cols))
    ]


def find_zero_cols(A):
    """Return the indices of exactly zero columns in matrix A."""
    return [
        j for j in range(A.cols)
        if all(A[i, j] == 0 for i in range(A.rows))
    ]

def smallest_nonzero_entries(A, n=10, dps=80):
    """Return the n smallest nonzero entries by numerical magnitude."""
    import mpmath as mpm

    entries = []

    with mpm.workdps(dps):
        for i in range(A.rows):
            for j in range(A.cols):
                value = A[i, j]

                if value != 0:
                    magnitude = abs(
                        mpm.mpf(str(value.n(dps)))
                    )
                    entries.append((magnitude, i, j, value))

        entries.sort(key=lambda item: item[0])

    return entries[:n]

# Specification of bodies and wires

In [4]:
# Ground specification
pos_GND = { 'for_F1' : utils.center( 0 ) }
GND_data = sus.GroundData( pos = pos_GND )

In [5]:
# GAS filter specification

# mass = 100.21
# f0 = 0.250
# saturation = -1e-3
# loss_angle = 1e-4
# moi = utils.matrix_3x3_diagonal( 3.670091 , 3.662444 , 6.095939 )
# pos_F1 = { 'top':utils.center( 1e-3 ) , 'bottom' : utils.center( -1e-3 ) }


mass = 91.31        # kg
f0 = 0.250          # Hz
saturation = -1e-3  # dimensionless
loss_angle = 1e-4   # dimensionless

moi = utils.matrix_3x3_diagonal( 3.670091 , 3.662444 , 6.095939 ) # kg⋅m²


# moi = utils.matrix_3x3( [ [ 3.8399, 0.61  , 0      ],
#                           [ 0.61  , 3.8399, 0      ],
#                           [ 0     , 0     , 6.4676 ] ] ) # kg⋅m²

# pos_F1 = { 'for_GND':utils.center( -13.5e-3 ),
#            'for_CB' : utils.center( -24.6e-3 ) } # args in m

pos_F1 = { 'for_GND':utils.center( 1e-3 ),
           'for_CB' : utils.center( -1e-3 ) } # args in m

F1_data = sus.GASFilterData( mass = mass,
                                    moi = moi,
                                    pos = pos_F1,
                                    resonance_frequency = f0,
                                    saturation = saturation,
                                    loss_angle = loss_angle,
                                    shape = shapes.GASFilterShape ) 

In [6]:
type( F1_data.loss_angle )

symengine.lib.symengine_wrapper.Rational

In [7]:
F1_data.mass

1606342507715625/17592186044416

In [8]:
# Mass specification
mass = 2
side = utils.con( 30e-3 )
moi = utils.con( 1 / 6 ) * mass * side**2 * utils.matrix_3x3_diagonal(1,1,1)
pos_CB = { 'for_F1' : utils.center( 3e-3 ) } 
CB_data = sus.BodyData( mass = mass ,
                        moi = moi ,
                        pos = pos_CB ,
                        shape = shapes.CubeShape )

In [9]:
print( type( CB_data.moi ) )
CB_data.moi

<class 'symengine.lib.symengine_wrapper.MutableDenseMatrix'>


[7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616, 0, 0]
[0, 7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616, 0]
[0, 0, 7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616]

In [10]:
# number = 1
# material = materials.maraging_steel
# diameter = 3e-3
# length = utils.con( 26.5e-3 + 1280e-3 + 26.5e-3 )
# wireF1_data = utils.WireData( number = number , material = material , diameter = diameter , length = length )

number = 1
material = materials.maraging_steel
diameter = ( 3e-3 , 4.5e-3 , 3e-3 )      # m
length = ( 26.5e-3 , 1280e-3 , 26.5e-3 ) # m

wireF1_data = sus.WireData( number = number,
                        material = material,
                        diameter = diameter,
                        length = length )

In [11]:
number = 1
material = materials.maraging_steel
diameter = 2.5e-3 
length = 26.5e-3 + 513.4e-3 + 26.5e-3
wireCB_data = sus.WireData( number = number , material = material , diameter = diameter , length = length )

# Declare bodies and wires and connect them

In [12]:
# Declare suspension
TS = sus.Suspension('TS')

In [13]:
# Declare body instances
GND = sus.Ground( 'GND' , data = GND_data )
F1 = sus.GASFilter( 'F1' , data = F1_data )
CB  = sus.RigidBody( 'CB' , data = CB_data )

F1: the attachment point at the lower end of the internal spring is called 'for_CB'.


In [14]:
# Declare link instances
wire_F1 = sus.Wire( data = wireF1_data )
wire_CB = sus.Wire( data = wireCB_data )

In [15]:
# Connect ground and cube with spring
TS.connect( GND , 'for_F1' , F1 , 'for_GND' , wire_F1 )
TS.connect( F1 , 'for_CB' , CB , 'for_F1' , wire_CB )
GND.show()

GND
└── F1
    └── CB


In [16]:
# print( F1.spring_body )

In [17]:
TS.update()

F1: beta, m1, m2 = -0.001 , 0.1 , 0.0360453484494832


In [18]:
F1.link_internal

In [19]:
info.show_info( TS , 'load')

╭────────┬─────────────┬─────────────┬────────────────┬───────────────╮
│ Name   │   Mass [kg] │   Load [kg] │      Number of │   Single link │
│        │             │             │   links in set │     load [kg] │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ GND    │             │       93.31 │                │               │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ F1     │       91.31 │        2    │              1 │         93.31 │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ CB     │        2    │        0    │              1 │          2    │
╰────────┴─────────────┴─────────────┴────────────────┴───────────────╯


In [20]:
info.show_info( TS , 'links')

╭────────┬──────────────┬────────────────┬───────────────┬─────────────┬─────────────╮
│ Name   │     Link set │      Number of │   Single link │       Axial │   Torsional │
│        │   total load │   links in set │     load [kg] │   stiffness │   stiffness │
│        │         [kg] │                │               │       [N/m] │    [Nm/rad] │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ GND    │              │                │               │             │             │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ F1     │        93.31 │              1 │         93.31 │ 2.21643e+06 │      5353.1 │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ CB     │         2    │              1 │          2    │ 1.68998e+06 │     24408   │
╰────────┴──────────────┴────────────────┴───────────────┴─────────────┴─────────────╯


In [52]:
F1.link.k_torsion.n(50)

5353.10360885206

In [53]:
CB.link.k_torsion.n(50)

24408.0426739088

In [21]:
print( TS.float_coords_list )

None


In [22]:
print( TS.equi_coords_guess_dict )

None


In [23]:
TS.calculate_equilibrium_positions(decimal_places= 150)

Let's calculate the equilibrium positions...
Guessing the values of the equilibrium coordinates... Done.
Calculating the total potential energy...
	Calculating the total wire axial elastic potential energy... Done.
	I'm reasonably assuming that the link elastic torsional potential energies are zero at equilibrium.
Calculating the gradient...
For this task: decimal_places = 150.
Minimizing the total potential energy... [mpf('0.0'), mpf('0.0'), mpf('0.0'), mpf('0.0'), mpf('-0.001000000000000000020816681711721685132943093776702880859375'), mpf('0.0'), mpf('0.0'), mpf('0.0'), mpf('0.0'), mpf('-1.90582445790259586243151811830690798669193101334969573197511412319683984079679310100793400298028602188200200748687714668697773655002463776009016007552943'), mpf('-1.33441285225428598220435407121455149399044844963321728287667997049772718943537346433091183062494024917963414359930853312265157692728926112873784050471321')]
Done.
Equilibrium positions found.
With knowledge of the equilibrium positions, I

In [24]:
info.print_equilibrium_positions(TS)

╭──────────┬─────────────┬──────────────╮   ╭──────────┬─────────────┬──────────────╮   ╭──────────┬─────────────┬──────────────╮
│ Symbol   │     Guessed │   Calculated │   │ Symbol   │     Guessed │   Calculated │   │ Symbol   │     Guessed │   Calculated │
│          │   value [m] │    value [m] │   │          │   value [m] │    value [m] │   │          │   value [m] │    value [m] │
├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤
│ lonCB    │       0     │        0     │   │ traCB    │     0       │      0       │   │ rollGND  │           0 │            0 │
├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤
│ lonF1    │       0     │        0     │   │ traF1    │     0       │      0       │   │ pitchGND │           0 │            0 │
├──────────┼─────────────┼──────────────┤   ├──────────┼─────────────┼──────────────┤   ├─

In [25]:
TS.equi_coords_dict

{lonCB: 0,
 lonF1: 0,
 pitchCB: 0,
 pitchF1: 0,
 qF1: -1152921504606847/1152921504606846976,
 rollCB: 0,
 rollF1: 0,
 traCB: 0,
 traF1: 0,
 verCB: -190582445790259586243151811830690798669193101334969573197511412319683984079679310100793400298028602188200200748687714668697773655002463776009016007553/100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,
 verF1: -133441285225428598220435407121455149399044844963321728287667997049772718943537346433091183062494024917963414359930853312265157692728926112873784050471/100000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000,
 lonGND: 0,
 traGND: 0,
 verGND: 0,
 rollGND: 0,
 pitchGND: 0,
 yawGND: 0,
 yawF1: 0,
 yawCB: 0}

In [26]:
print( F1.link.Ep_dict['torsion']['energy'].n(20) )
print( CB.link.Ep_dict['torsion']['energy'].n(20) )

2676.55180442603*(-1.0*yawF1 + yawGND)**2.0
12204.0213369544*(-1.0*yawCB + yawF1)**2.0


In [27]:
print( F1.link.Ep_dict['axial']['energy'].n(20) )
print()
print( CB.link.Ep_dict['axial']['energy'].n(20) )

1108215.47175833*(-1.333 + ((lonGND - 1.0*(lonF1 + 0.001*(sin(yawF1)*sin(rollF1) + cos(yawF1)*cos(rollF1)*sin(pitchF1))))**2.0 + (verGND - 1.0*(verF1 + 0.001*cos(rollF1)*cos(pitchF1)))**2.0 + (traGND - 1.0*(traF1 + 0.001*(-1.0*cos(yawF1)*sin(rollF1) + sin(yawF1)*cos(rollF1)*sin(pitchF1))))**2.0)**0.5)**2.0

844989.41705565*(-0.5664 + ((lonF1 + qF1*(sin(yawF1)*sin(rollF1) + cos(yawF1)*cos(rollF1)*sin(pitchF1)) - 1.0*(lonCB + 0.003*(sin(yawCB)*sin(rollCB) + cos(yawCB)*cos(rollCB)*sin(pitchCB))) - 0.001*(sin(yawF1)*sin(rollF1) + cos(yawF1)*cos(rollF1)*sin(pitchF1)))**2.0 + (traF1 + qF1*(-1.0*cos(yawF1)*sin(rollF1) + sin(yawF1)*cos(rollF1)*sin(pitchF1)) - 1.0*(traCB + 0.003*(-1.0*cos(yawCB)*sin(rollCB) + sin(yawCB)*cos(rollCB)*sin(pitchCB))) - 0.001*(-1.0*cos(yawF1)*sin(rollF1) + sin(yawF1)*cos(rollF1)*sin(pitchF1)))**2.0 + (verF1 - 0.001*cos(rollF1)*cos(pitchF1) + qF1*cos(rollF1)*cos(pitchF1) - 1.0*(verCB + 0.003*cos(rollCB)*cos(pitchCB)))**2.0)**0.5)**2.0


In [28]:
print( TS.float_coords_list )

None


In [29]:
TS.calculate_mass_matrix()

Let's calculate the mass matrix...
Compiling position and velocity coordinates...Done.
	Number of body coordinates: 13
	Number of float coordinates: 0
	Number of ground coordinates: 6
calculating matrix... Done.
Elapsed time: 0.0063 seconds.
Done, call TS.M_matrix to see the result.


In [30]:
print( TS.float_coords_list )

[]


In [31]:
TS.calculate_stiffness_matrix( with_damping = False )

Let's calculate the master stiffness matrix and its submatrices...
The setting `with_damping = False` is suitable for later calculating eigenvectors,
for calculating transfer functions set `with_damping = True`.
For F1:
	Link axial potential energy: calculating matrix... Done.
	Link torsional potential energy: calculating matrix... Done.
	Adding these matrices to total... Done.
	Link axial potential energy: calculating matrix... Done.
	Link torsional potential energy: calculating matrix... Done.
	Adding these matrices to total... Done.
For CB:
	Link axial potential energy: calculating matrix... Done.
	Link torsional potential energy: calculating matrix... Done.
	Adding these matrices to total... Done.
I have the final master stiffness matrix, you can see it by calling TS.P
Extracting the relevant submatrices from the master stiffness matrix...
	There are no float coordinates, matrices Q, C_XQ, C_QS and related don't exist.
Finished, elapsed time: 0.0191 seconds.


In [32]:
print( TS.P.shape )
print( TS.M_matrix.shape )

(19, 19)
(13, 13)


In [33]:
print( is_symmetric( TS.M_matrix ) )
print( is_symmetric( TS.P ) )

True
True


In [34]:
zero_rows = find_zero_rows(TS.P)
zero_cols = find_zero_cols(TS.P)

print("Zero rows:", zero_rows)
print("Zero columns:", zero_cols)

Zero rows: [16, 17]
Zero columns: [16, 17]


In [35]:
for magnitude, i, j, value in smallest_nonzero_entries(TS.P):
    print(f"K[{i}, {j}] = {value}")
    print(f"    |K[{i}, {j}]| ≈ {magnitude}")

K[3, 10] = (1084677219512208605642065626302628841476747064011399267072742035792022870693950092496890969471612619838841494561267398097482002582316027121174886148780874116008385555193314907121360066217853581806906285422607421875/16401388414130984630537523962535186765453186303217788384447309435866431549533200871856485227678213150294878484127666672883208182826204745731184385659037965359368342431756682002154627567715281291001247826006273008074752)*pi
    |K[3, 10]| ≈ 0.000207763751354114125612889999999999999999999999999999999999999999999999999999999999980855039961561808750669096844834869595134791755757859773052842715898
K[4, 11] = (1084677219512208605642065626302628841476747064011399267072742035792022870693950092496890969471612619838841494561267398097482002582316027121174886148780874116008385555193314907121360066217853581806906285422607421875/164013884141309846305375239625351867654531863032177883844473094358664315495332008718564852276782131502948784841276666728832081828262047457311843856590

In [36]:
TS.calculate_eigenvectors( TS.M_matrix )

Shape: (13, 13)
For this task: decimal_places = 150.
Number of modes: 13
eigenvalues: [0.0003, 0.0003, 0.0003, 0.0340002, 2.0, 2.0, 2.0, 3.66244, 3.67009, 6.09594, 91.446, 91.446, 91.4461]
Finished, elapsed time: 0.0056 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)


In [37]:
# TS.calculate_eigenvectors( TS.P )

In [38]:
# TS.calculate_eigenvectors()
# TS.calculate_eigenvectors(decimal_places = 80)

In [39]:
for prec in (50, 80, 100):
    TS.calculate_eigenvectors(TS.P, decimal_places = prec)
    print('\n')
    # for key, value in TS.eigen_dict.items():
    #     print( key, mpm.nstr( value['frequency'],6 ) )
    print('\n')
    print('\n')
    

Shape: (19, 19)
For this task: decimal_places = 50.
Number of modes: 19
eigenvalues: [-2.33317e-43, -1.4813e-45, 5.83393e-48, 1.12013e-47, 3.32792e-46, 8.4253e-45, 0.0588395, 0.0588395, 0.954281, 0.954281, 3.01123, 51.2696, 51.2696, 1390.49, 1390.49, 7540.67, 51981.6, 2.78998e+6, 6.71282e+6]
Imaginary frequency detected: 1, (0.0 + 7.6876e-23j).
Imaginary frequency detected: 2, (0.0 + 6.1255e-24j).
Finished, elapsed time: 0.0848 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)






Shape: (19, 19)
For this task: decimal_places = 80.
Number of modes: 19
eigenvalues: [-2.33333e-43, -5.45375e-75, 1.04178e-112, 4.28961e-75, 3.33333e-46, 3.33333e-46, 0.0588395, 0.0588395, 0.954281, 0.954281, 3.01123, 51.2696, 51.2696, 1390.49, 1390.49, 7540.67, 51981.6, 2.78998e+6, 6.71282e+6]
Imaginary frequency detected: 1, (0.0 + 7.6879e-23j).
Imaginary frequency detected: 2, (0.0 + 1.1754e-38j).
Finished, elapsed time: 0.0858 seconds.
You can see the eigenfrequency list b

In [40]:
info.print_eigenfreq(TS)

╭──────────┬─────────────╮   ╭──────────┬─────────────╮   ╭──────────┬─────────────╮
│     Mode │   Frequency │   │     Mode │   Frequency │   │     Mode │   Frequency │
│   number │        [Hz] │   │   number │        [Hz] │   │   number │        [Hz] │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        1 │ 7.68791e-23 │   │        8 │    0.038606 │   │       15 │     5.93478 │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        2 │ 4.89107e-49 │   │        9 │    0.155474 │   │       16 │    13.8205  │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        3 │ 1.2145e-77  │   │       10 │    0.155474 │   │       17 │    36.2865  │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        4 │ 7.40773e-49 │   │       11 │    0.27618  │   │       18 │   265.84    │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────

In [41]:
n = 1
info.print_eigen( TS , n )

Mode number: 1 

Frequency: 0.0 Hz

╭──────────────┬───────────────╮   ╭──────────────┬───────────────╮   ╭──────────────┬───────────────╮
│ Coordinate   │         Value │   │ Coordinate   │         Value │   │ Coordinate   │         Value │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ├──────────────┼───────────────┤
│ lonF1        │ -5.2948e-211  │   │ yawF1        │  1            │   │ rollCB       │  5.28824e-260 │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ├──────────────┼───────────────┤
│ traF1        │ -6.57504e-215 │   │ qF1          │ -1.96983e-252 │   │ pitchCB      │ -2.08411e-252 │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ├──────────────┼───────────────┤
│ verF1        │  1.24373e-210 │   │ lonCB        │ -5.2948e-211  │   │ yawCB        │  1            │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ╰──────────────┴───────────────╯
│ rollF1       │ -1.0738e-259  │   │ 

In [42]:
TS.coords_list

[lonF1,
 traF1,
 verF1,
 rollF1,
 pitchF1,
 yawF1,
 qF1,
 lonCB,
 traCB,
 verCB,
 rollCB,
 pitchCB,
 yawCB,
 lonGND,
 traGND,
 verGND,
 rollGND,
 pitchGND,
 yawGND]

In [43]:
with mpm.workdps(100):
    for i in [5, 12]:
        row = []
        for j in [5, 12]:
            value = TS.P[i, j]
            value_mp = mpm.mpf(str(value.n(mpm.mp.dps)))
            row.append(mpm.nstr(value_mp, 30))
        print(row)

['29761.14628276087277223628445', '-24408.042673908813215118177356']
['-24408.042673908813215118177356', '24408.042673908813215118177356']


In [44]:
for prec in (50, 80, 100):
    TS.calculate_eigenvectors(decimal_places = prec)
    print('\n')
    for key, value in TS.eigen_dict.items():
        print( key, mpm.nstr( value['frequency'],6 ) )
    print('\n')
    print('\n')
    

Shape: (13, 13)
For this task: decimal_places = 50.
Number of modes: 13
eigenvalues: [0.260009, 0.260552, 2.46739, 7.23198, 7.23198, 17.8626, 17.8626, 197.274, 197.274, 878.099, 24245.6, 5.05339e+7, 8.13641e+7]
Finished, elapsed time: 0.0402 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)


1 0.0811548
2 0.0812395
3 0.249999
4 0.428005
5 0.428005
6 0.672655
7 0.672655
8 2.2354
9 2.2354
10 4.7162
11 24.782
12 1131.39
13 1435.61




Shape: (13, 13)
For this task: decimal_places = 80.
Number of modes: 13
eigenvalues: [0.260009, 0.260552, 2.46739, 7.23198, 7.23198, 17.8626, 17.8626, 197.274, 197.274, 878.099, 24245.6, 5.05339e+7, 8.13641e+7]
Finished, elapsed time: 0.0443 seconds.
You can see the eigenfrequency list by calling info.print_eigenfreq(TS)


1 0.0811548
2 0.0812395
3 0.249999
4 0.428005
5 0.428005
6 0.672655
7 0.672655
8 2.2354
9 2.2354
10 4.7162
11 24.782
12 1131.39
13 1435.61




Shape: (13, 13)
For this task: decimal_places = 100.
Number of m

In [45]:
info.print_eigenfreq( TS )

╭──────────┬─────────────╮   ╭──────────┬─────────────╮   ╭──────────┬─────────────╮
│     Mode │   Frequency │   │     Mode │   Frequency │   │     Mode │   Frequency │
│   number │        [Hz] │   │   number │        [Hz] │   │   number │        [Hz] │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        1 │   0.0811548 │   │        6 │    0.672655 │   │       11 │      24.782 │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        2 │   0.0812395 │   │        7 │    0.672655 │   │       12 │    1131.39  │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ├──────────┼─────────────┤
│        3 │   0.249999  │   │        8 │    2.2354   │   │       13 │    1435.61  │
├──────────┼─────────────┤   ├──────────┼─────────────┤   ╰──────────┴─────────────╯
│        4 │   0.428005  │   │        9 │    2.2354   │
├──────────┼─────────────┤   ├──────────┼─────────────┤
│        5 │   0.428005  │   │       1

In [46]:
n = 2
info.print_eigen( TS , n )

Mode number: 2 

Frequency: 0.081 Hz

╭──────────────┬───────────────╮   ╭──────────────┬───────────────╮   ╭──────────────┬───────────────╮
│ Coordinate   │         Value │   │ Coordinate   │         Value │   │ Coordinate   │         Value │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ├──────────────┼───────────────┤
│ lonF1        │  0.00103841   │   │ yawF1        │ -1.70979e-105 │   │ rollCB       │ -2.05945e-97  │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ├──────────────┼───────────────┤
│ traF1        │  2.93965e-100 │   │ qF1          │ -1.53115e-96  │   │ pitchCB      │ -8.20761e-05  │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ├──────────────┼───────────────┤
│ verF1        │ -2.75696e-102 │   │ lonCB        │  0.00308508   │   │ yawCB        │ -1.70979e-105 │
├──────────────┼───────────────┤   ├──────────────┼───────────────┤   ╰──────────────┴───────────────╯
│ rollF1       │  6.16185e-96  │   

In [47]:
assert False, "Stopped execution here"

AssertionError: Stopped execution here

In [ ]:
anime.prepare_anime( TS )

In [ ]:
n = 7
# info.print_eigen( TS , n )
# anime.show_anime( TS , n )

In [ ]:
print( F1.spring_load )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
TS.calculate_stiffness_matrix( with_damping = True )

In [ ]:
TS.prepare_transfer_functions()

In [ ]:
frequency = np.logspace(-2,4,1000)

In [ ]:
TS.body_coords_list

In [ ]:
in_coord = GND.ver
out_coord = CB.ver
tf = TS.transfer_function( in_coord , out_coord )

In [ ]:
plt.loglog( frequency , np.abs( tf( 2*np.pi*frequency ) ) )
plt.axvline(x=0.25, color='r', linestyle='--', linewidth=2)
plt.axhline(y=1e-3, color='r', linestyle='--')

In [ ]:
7771980404306739/274877906944

In [ ]:
import symengine

# Create a 0x1 dense matrix (vector)
empty_vec = symengine.Matrix(1, 0, []).T

# Convert to list
result = empty_vec.T.tolist()

print(result)  # Output: []
print(type(result))  # Output: <class 'list'>


In [ ]:
import symengine
from symengine import DenseMatrix

# Create a 0x1 empty column vector
v = DenseMatrix(1, 0, []).T 

# Transpose it to a 1x0 row vector
v_T = v.T

# Convert to list: results in a nested empty list [[]]
result = v_T.tolist()

print(f"Original: {v.tolist()}") # Output: []
print(f"Transposed: {result}")   # Output: [[]]
print(f"Shape: {v_T.shape}")      # Output: (1, 0)


In [ ]:
import symengine as se

In [ ]:
isinstance( utils.con( 5 , 5 , 3 ) , se.Number )

In [ ]:
utils.con( 30 , 43 , 6 )

In [ ]:
F1.omega

In [ ]:
F1.omega[2]

In [ ]:
my_keys = GND.gnd_coords
my_values = [ utils.con(0) ] * len( my_keys )
dict( zip( my_keys , my_values ) ) 

In [ ]:
aa = [1.23 , 6.43]

In [ ]:
bb = [aa] if not isinstance( aa , ( list , tuple ) ) else aa

In [ ]:
bb

In [ ]:
utils.con( 1.23 , 6.43 )

In [ ]:
utils.con( 1.23 )